In [1]:
import pandas as pd
!pip freeze | grep pandas

geopandas==1.1.1
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
sklearn-pandas==2.2.0


In [2]:
# financials 
# in millions
year = [2020, 2021, 2022, 2023, 2024]
revenue = [37402, 44500, 46710, 51217, 51362]
cost_of_goods_sold = [21159, 24550, 25231, 28925, 28475]
operating_income = [3130, 6910, 6311, 5915, 6311]
depreciation_and_amortization = [600, 650, 840, 859, 844]
pretax_income = [2840, 6630, 6651, 6201, 6700]
tax = [344, 930, 605, 1131, 1000]
net_income = [2496, 5730, 6046, 5070, 5700]
operating_cash_flow = [3698, 7429, 5188, 5841, 7429]
capex = [718, 500, 758, 969, 812]
total_debt = 1000 + 7903
total_shares_outstanding = 1503

In [3]:
# assumptions
ocf_growth = 0.02
capex_growth = 0.02
discount_rate = 0.10
forecast_years = 5
terminal_growth = 0.01

In [4]:
df = pd.DataFrame(columns=year + [x+1 for x in range(year[-1], year[-1]+forecast_years)])
df.loc["revenue"] = revenue + [None]*forecast_years
df.loc["cost_of_goods_sold"] = cost_of_goods_sold + [None]*forecast_years
df.loc["operating_income"] = operating_income + [None]*forecast_years
df.loc["depreciation_and_amortization"] = depreciation_and_amortization + [None]*forecast_years
df.loc["pretax_income"] = pretax_income + [None]*forecast_years
df.loc["tax"] = tax + [None]*forecast_years
df.loc["net_income"] = net_income + [None]*forecast_years
df.loc["operating_cash_flow"] = operating_cash_flow + [operating_cash_flow[-1]*(1 + ocf_growth)**x for x in range(1, forecast_years+1)]
df.loc["capex"] = capex + [capex[-1]*(1 + capex_growth)**x for x in range(1, forecast_years+1)]
df.loc["free_cash_flow"] = df.loc["operating_cash_flow"] - df.loc["capex"]
df.loc["discount_factor"] = [None]*len(revenue) + [1 / ((1 + discount_rate) ** x) for x in range(1, forecast_years+1)]
df.loc["pv_free_cash_flow"] = df.loc["free_cash_flow"] * df.loc["discount_factor"]
df.loc["terminal_value", year[-1]+forecast_years] = df.loc["free_cash_flow", year[-1]+forecast_years] * (1 + terminal_growth) / (discount_rate - terminal_growth)
df.loc["pv_terminal_value"] = df.loc["terminal_value"] * df.loc["discount_factor"]
df.loc["enterprise_value", year[-1]] = df.loc["pv_free_cash_flow"].sum() + df.loc["pv_terminal_value"].sum()
df.loc["total_debt", year[-1]] = total_debt
df.loc["equity_value"] = df.loc["enterprise_value"] - df.loc["total_debt"]
df.loc["total_shares_outstanding", year[-1]] = total_shares_outstanding
df.loc["equity_value_per_share"] = df.loc["equity_value"] / df.loc["total_shares_outstanding"]
df.round(2)

,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
revenue,37402.0,44500.0,46710.0,51217.0,51362.00,NaN,NaN,NaN,NaN,NaN
cost_of_goods_sold,21159.0,24550.0,25231.0,28925.0,28475.00,NaN,NaN,NaN,NaN,NaN
operating_income,3130.0,6910.0,6311.0,5915.0,6311.00,NaN,NaN,NaN,NaN,NaN
depreciation_and_amortization,600.0,650.0,840.0,859.0,844.00,NaN,NaN,NaN,NaN,NaN
pretax_income,2840.0,6630.0,6651.0,6201.0,6700.00,NaN,NaN,NaN,NaN,NaN
tax,344.0,930.0,605.0,1131.0,1000.00,NaN,NaN,NaN,NaN,NaN
net_income,2496.0,5730.0,6046.0,5070.0,5700.00,NaN,NaN,NaN,NaN,NaN
operating_cash_flow,3698.0,7429.0,5188.0,5841.0,7429.00,7577.58,7729.13,7883.71,8041.39,8202.22
capex,718.0,500.0,758.0,969.0,812.00,828.24,844.80,861.70,878.93,896.51
free_cash_flow,2980.0,6929.0,4430.0,4872.0,6617.00,6749.34,6884.33,7022.01,7162.45,7305.70
